# Transformer-Based News Classification

## Overview

This notebook uses DistilBERT to classify news articles into four categories from the AG News dataset.

It covers dataset preparation, text tokenization, model fine-tuning, evaluation, and inference on a new news article.

### Objectives

- Prepare the AG News dataset for transformer-based classification.
- Load a pre-trained DistilBERT model and tokenizer.
- Tokenize the text and prepare the datasets.
- Fine-tune DistilBERT using the Hugging Face Trainer API.
- Evaluate the model using accuracy and macro F1-score.
- Generate predictions for a new news article.

### News Categories

The AG News dataset contains four categories:

- **World**
- **Sports**
- **Business**
- **Sci/Tech**

In [ ]:
# 1. Install required libraries
%pip install -q transformers datasets accelerate scikit-learn

## 1. Loading the AG News Dataset

We use the AG News dataset from the Hugging Face Datasets library. It contains news articles belonging to four categories: World, Sports, Business, and Sci/Tech.

The dataset provides separate training and test splits. We load the dataset and inspect its structure, the number of samples, and an example article with its corresponding label.

This initial inspection helps us understand the data before preparing it for model training.

In [ ]:
from datasets import load_dataset

dataset = load_dataset("fancyzhx/ag_news")

print("Available splits:", list(dataset.keys()))
print("Training samples:", len(dataset["train"]))
print("Test samples:", len(dataset["test"]))
print("First news:", dataset["train"][0]["text"])
print("First label:", dataset["train"][0]["label"])

README.md:   0%|          | 0.00/8.07k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 18.6MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

data/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.23MB            

data/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/120000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/7600 [00:00<?, ? examples/s]

Available splits: ['train', 'test']
Training samples: 120000
Test samples: 7600
First news: Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
First label: 2


## 2. Splitting the Dataset

The original training set is divided into two subsets:

- **Training set:** Used to fine-tune the model.
- **Validation set:** Used to evaluate the model during training and select the best checkpoint.

We reserve 10% of the original training data for validation. The `stratify` parameter preserves the approximate distribution of news categories in both subsets, while `random_state=42` makes the split reproducible.

The official test set is kept separate and is not used during training or checkpoint selection.

In [ ]:
from sklearn.model_selection import train_test_split

all_train_texts = dataset["train"]["text"]
all_train_labels = dataset["train"]["label"]

train_texts, val_texts, train_labels, val_labels = train_test_split(
    all_train_texts,
    all_train_labels,
    test_size=0.1,
    random_state=42,
    stratify=all_train_labels,
)

test_texts = dataset["test"]["text"]
test_labels = dataset["test"]["label"]

print("Training samples:", len(train_texts))
print("Validation samples:", len(val_texts))
print("Test samples:", len(test_texts))

## 3. Checking the Computing Environment

Before loading and training the transformer model, we check the installed PyTorch version and whether a GPU is available.

GPU acceleration can significantly reduce training time for transformer models. If a compatible GPU is available, PyTorch can use it for computation; otherwise, the model can run on the CPU, although training may take longer.

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch version: 2.11.0+cu130
GPU available: True
GPU: Tesla T4


## 4. Loading the Pre-trained DistilBERT Model

DistilBERT is a smaller and more computationally efficient version of BERT.

We load a pre-trained DistilBERT model and its tokenizer. The tokenizer converts raw text into tokens that the model can process, while the pre-trained model provides language representations learned from a large text corpus.

We set `num_labels=4` because the AG News dataset contains four news categories. The maximum sequence length is set to 128 tokens to limit input length and manage computational requirements.

The model will be fine-tuned on the AG News training dataset in the following sections.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=4,
)

print("Tokenizer and model loaded successfully.")
print("Number of labels:", model.config.num_labels)

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  268MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer and model loaded successfully.
Number of labels: 4


## 5. Tokenization and Dataset Preparation

Transformer models cannot process raw text directly. The tokenizer converts each news article into numerical token IDs and creates attention masks that help DistilBERT identify the relevant tokens.

We define a tokenization function that truncates articles to a maximum length of 128 tokens. This helps control memory usage and computational cost.

The Hugging Face `Dataset` objects are created for the training, validation, and test sets. After tokenization, the original text column is removed because the model uses the tokenized inputs and labels.

Finally, `DataCollatorWithPadding` dynamically pads sequences within each batch. This avoids padding every article to the same length in advance and helps use computational resources more efficiently.

In [ ]:
from datasets import Dataset
from transformers import DataCollatorWithPadding

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_dataset = Dataset.from_dict({
    "text": list(train_texts),
    "label": list(train_labels),
}).map(tokenize_function, batched=True)

val_dataset = Dataset.from_dict({
    "text": list(val_texts),
    "label": list(val_labels),
}).map(tokenize_function, batched=True)

test_dataset = Dataset.from_dict({
    "text": list(test_texts),
    "label": list(test_labels),
}).map(tokenize_function, batched=True)

train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Training samples:", len(train_dataset))
print("Validation samples:", len(val_dataset))
print("Test samples:", len(test_dataset))
print("Tokenized columns:", train_dataset.column_names)

## 6. Defining Evaluation Metrics

We define two metrics to evaluate the classification model:

- **Accuracy:** The proportion of articles classified correctly across all categories.
- **Macro F1-score:** The unweighted average of the F1-scores calculated separately for each category.

Accuracy provides an overall measure of classification performance. Macro F1 gives equal importance to all four categories, regardless of how many examples belong to each category.

The `compute_metrics` function converts the model's output logits into predicted class labels using `argmax` and then calculates both metrics.

In [ ]:
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "macro_f1": f1_score(
            labels,
            predictions,
            average="macro",
        ),
    }

## 7. Configuring the Training Process

We define the training configuration using Hugging Face's `TrainingArguments`.

The main settings include:

- **Number of epochs:** The model passes through the training dataset three times.
- **Learning rate:** A value of `2e-5` controls the size of the model's parameter updates.
- **Batch sizes:** The model processes 16 training examples and 32 evaluation examples per batch.
- **Weight decay:** A value of `0.01` helps regularize the model.
- **Evaluation and checkpointing:** The model is evaluated and saved at the end of each epoch.
- **Best model selection:** The checkpoint with the highest validation accuracy is selected.
- **Mixed-precision training:** FP16 is enabled when a CUDA-compatible GPU is available.

We also set a random seed to improve reproducibility and limit the number of saved checkpoints to reduce storage usage.

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="outputs/distilbert-ag-news-checkpoints",
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
    save_total_limit=1,
    seed=42,
)

## 8. Initializing the Trainer

The Hugging Face `Trainer` API simplifies the model training and evaluation process.

We provide the following components:

- **Model:** The pre-trained DistilBERT model that will be fine-tuned.
- **Training arguments:** The configuration defined in the previous section.
- **Training dataset:** The tokenized articles used to update the model's parameters.
- **Validation dataset:** The data used to evaluate the model during training.
- **Data collator:** Dynamically pads sequences within each batch.
- **Evaluation function:** Computes accuracy and macro F1-score.

Once initialized, the Trainer manages the training loop, evaluation, and checkpoint saving according to the specified configuration.

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

print("Trainer initialized successfully.")

Trainer initialized successfully.


## 9. Fine-Tuning DistilBERT

We fine-tune the pre-trained DistilBERT model on the AG News training dataset.

During training, the model updates its parameters to improve its ability to classify news articles into the four target categories.

The Trainer follows the configuration defined earlier, including the number of epochs, learning rate, batch size, and checkpoint-saving strategy.

Training can take some time, depending on the available computing resources.

In [ ]:
training_result = trainer.train()

## 10. Evaluating the Fine-Tuned Model

After training, we evaluate the model on the validation dataset using the Hugging Face Trainer.

We inspect the evaluation metrics, including accuracy and macro F1-score, to assess the model's classification performance.

The Trainer also reports the best checkpoint selected during training according to validation accuracy. This checkpoint can be used for subsequent inference.

The validation set is used for model selection, while the official test set remains separate for final evaluation.

In [ ]:
validation_results = trainer.evaluate()

print("Validation results:")
print(validation_results)

In [ ]:
print("Training completed.")
print("Best model checkpoint:", trainer.state.best_model_checkpoint)
print("Best validation metric:", trainer.state.best_metric)

In [ ]:
id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}

label2id = {
    label: label_id
    for label_id, label in id2label.items()
}

print("AG News label mapping:")

for label_id, label_name in id2label.items():
    print(f"{label_id}: {label_name}")

In [ ]:
# Make sure the trained model is ready for inference
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = model.to(device)
model.eval()

# Example news article
text = "The stock market rose after the government announced a new economic policy."

# Convert the text into model inputs
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
).to(device)

# Predict without calculating gradients
with torch.inference_mode():
    outputs = model(**inputs)

# Get the predicted class
predicted_id = outputs.logits.argmax(dim=-1).item()

# Map the class ID to its news category
predicted_label = id2label[predicted_id]

# Calculate the predicted class probability
probabilities = torch.softmax(outputs.logits, dim=-1)[0]
confidence = probabilities[predicted_id].item()

print("News:", text)
print("Predicted ID:", predicted_id)
print("Predicted category:", predicted_label)
print(f"Confidence: {confidence:.4f}")

## 11. Inspecting the Saved Checkpoint

During training, the Trainer saves model checkpoints according to the configured saving strategy.

In this section, we check whether the expected checkpoint directory exists and inspect its contents. These files may include the model weights, model configuration, and tokenizer-related files required to reload the model.

This check helps verify that the saved checkpoint is available before attempting to load it for inference.

In [ ]:
from pathlib import Path

# Use the checkpoint directory defined in TrainingArguments
checkpoint_dir = Path(training_args.output_dir)

print("Checkpoint directory:", checkpoint_dir)
print("Checkpoint directory exists:", checkpoint_dir.exists())

if checkpoint_dir.exists():
    checkpoints = sorted(
        path
        for path in checkpoint_dir.glob("checkpoint-*")
        if path.is_dir()
    )

    if checkpoints:
        print("Available checkpoints:")

        for checkpoint in checkpoints:
            print(checkpoint.name)
    else:
        print("No checkpoints found.")
else:
    print("Checkpoint directory does not exist.")

## 12. Loading the Fine-Tuned Model for Inference

We load the fine-tuned DistilBERT model and its tokenizer from the Hugging Face Hub.

This allows us to use the trained model for inference without running the training process again.

The model is moved to the available computing device (GPU or CPU) and set to evaluation mode before making predictions.

In [2]:
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
)

MODEL_NAME = "RasulAramian/distilbert-ag-news"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

transformer_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
transformer_model = transformer_model.to(device)
transformer_model.eval()

print("Model loaded successfully.")
print("Device:", device)
print("Number of labels:", transformer_model.config.num_labels)
print("Labels:", transformer_model.config.id2label)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Error while downloading from https://huggingface.co/RasulAramian/distilbert-ag-news/resolve/1ae4b9c3fb14bb432ae5f502677426a07eb71c08/model.safetensors: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/RasulAramian/distilbert-ag-news/resolve/1ae4b9c3fb14bb432ae5f502677426a07eb71c08/model.safetensors: [Errno 104] Connection reset by peer
Trying to resume download...
Error while downloading from https://huggingface.co/RasulAramian/distilbert-ag-news/resolve/1ae4b9c3fb14bb432ae5f502677426a07eb71c08/model.safetensors: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/RasulAramian/distilbert-ag-news/resolve/1ae4b9c3fb14bb432ae5f502677426a07eb71c08/model.safetensors: The read operation timed out
Trying to resume download...
Error while downloading from https://huggingface.co/RasulAramian/distilbert-ag-news/resolve/1ae4b9c3fb14bb432ae5f502677426a07eb71c08/model.safetensors: 

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

Model loaded successfully.
Device: cpu
Number of labels: 4
Labels: {0: 'LABEL_0', 1: 'LABEL_1', 2: 'LABEL_2', 3: 'LABEL_3'}


## 13. Predicting the Category of a New Article

We test the fine-tuned model on a new news article that was not part of the training examples shown in this notebook.

The tokenizer converts the article into model inputs. The model produces logits for the four news categories, and `argmax` selects the category with the highest logit.

This demonstrates how the trained model can be used for inference on new text.

In [5]:
import torch

# Use the model loaded from Hugging Face
transformer_model.eval()

text = "The stock market rose after the government announced a new economic policy."

# Maximum number of tokens processed by the model
MAX_LENGTH = 128

# Tokenize the article and move inputs to the model's device
inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    max_length=MAX_LENGTH,
).to(device)

# Run inference without calculating gradients
with torch.inference_mode():
    outputs = transformer_model(**inputs)

# Get the predicted class
predicted_id = outputs.logits.argmax(dim=-1).item()

# Convert the class ID to its category name
id2label = {
    0: "World",
    1: "Sports",
    2: "Business",
    3: "Sci/Tech",
}

predicted_label = id2label[predicted_id]

# Calculate the probability of the predicted class
probabilities = torch.softmax(outputs.logits, dim=-1)[0]
confidence = probabilities[predicted_id].item()

print("News:", text)
print("Predicted ID:", predicted_id)
print("Predicted category:", predicted_label)
print(f"Confidence: {confidence:.4f}")

News: The stock market rose after the government announced a new economic policy.
Predicted ID: 2
Predicted category: Business
Confidence: 0.9956


## 14. Inspecting the Model Configuration and Training Data

In this section, we inspect the model's label mapping and configuration, along with the features of the tokenized training dataset.

These checks help us understand how the four output classes are represented and what information is available in the prepared training data.

In [ ]:
print("Model label mapping:")
print(transformer_model.config.id2label)

print("\nModel configuration:")
print(transformer_model.config)

In [ ]:
print(trainer.label_names)
print(trainer.train_dataset.features)

In [ ]:
print("Training dataset features:")
print(trainer.train_dataset.features)

print("\nFirst five training examples:")
for i in range(5):
    print(trainer.train_dataset[i])

## 15. Conclusion

In this notebook, we prepared the AG News dataset for transformer-based text classification and fine-tuned a pre-trained DistilBERT model.

The workflow included:

- Loading and splitting the dataset into training, validation, and test sets.
- Tokenizing news articles and preparing them for DistilBERT.
- Defining evaluation metrics, including accuracy and macro F1-score.
- Configuring and running the fine-tuning process using the Hugging Face Trainer API.
- Evaluating the model on the validation set and inspecting the saved checkpoint.
- Loading the fine-tuned model and using it to classify a new news article.

This notebook establishes the transformer-based approach that will be compared with the TF-IDF and Logistic Regression baseline in the next stage of the project.